In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import time
import copy
import os

# Verify hardware accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

if device.type == 'cpu':
    print("WARNING: GPU is not attached. Go to 'Runtime' > 'Change runtime type' > select 'T4 GPU' before proceeding.")
else:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Active Device: cuda
GPU Name: Tesla T4


In [3]:
import os

# Create target directories
!mkdir -p /content/Classifier /content/Severity

# Replace these zip names if yours are named differently (e.g., Dataset1.zip, Dataset2.zip)
!unzip -q /content/Classifier.zip -d /content/Classifier/
!unzip -q /content/Severity.zip -d /content/Severity/

# Verify the extracted contents
print("Classifier directory contents:", os.listdir('/content/Classifier'))
print("Severity directory contents:", os.listdir('/content/Severity'))

Classifier directory contents: ['Dataset1']
Severity directory contents: ['Severity']


In [4]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import os

# 1. Define Image Transformations
# Training includes data augmentation (flipping, rotating) to prevent overfitting
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation only resizes and normalizes (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Helper function to create Train/Val DataLoaders properly
def create_dataloaders(data_dir, batch_size=32, train_split=0.8):
    # Load dataset twice to isolate transforms
    train_dataset_full = datasets.ImageFolder(root=data_dir, transform=train_transforms)
    val_dataset_full = datasets.ImageFolder(root=data_dir, transform=val_transforms)

    # Calculate split sizes
    train_size = int(train_split * len(train_dataset_full))

    # Generate a fixed set of random indices for consistent splitting
    torch.manual_seed(42)
    indices = torch.randperm(len(train_dataset_full)).tolist()

    # Assign indices to subsets
    train_subset = Subset(train_dataset_full, indices[:train_size])
    val_subset = Subset(val_dataset_full, indices[train_size:])

    # Create DataLoaders
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, train_dataset_full.classes

# 3. Initialize DataLoaders using the corrected nested directory names
STAGE1_DIR = '/content/Classifier/Dataset1'  # Eczema vs Melanoma vs Psoriasis
STAGE2_DIR = '/content/Severity/Severity'    # Low vs Moderate vs High

gatekeeper_train_loader, gatekeeper_val_loader, gatekeeper_classes = create_dataloaders(STAGE1_DIR)
severity_train_loader, severity_val_loader, severity_classes = create_dataloaders(STAGE2_DIR)

print(f"Classifier Classes (Model 1): {gatekeeper_classes}")
print(f"Severity Classes (Model 2): {severity_classes}")

Classifier Classes (Model 1): ['Eczema', 'Melanoma', 'Psoriasis']
Severity Classes (Model 2): ['Atopic dermatitis childhood phase', 'Atopic dermatitis feet', 'Dyshidrosis', 'Eczema areola', 'Eczema asteatotic', 'Eczema chronic', 'Eczema fingertips', 'Eczema foot', 'Eczema hand', 'Ichthyosis', 'Keratolysis exfoliativa', 'Keratosis pilaris', 'Lichen simplex chronicus', 'Neurotic excoriations', 'Pompholyx', 'Prurigo nodularis', 'Stasis dermatitis']


In [5]:
import torchvision.models as models
import torch.nn as nn

def build_model(num_classes):
    # 1. Load the pre-trained MobileNetV2 architecture
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    # 2. Freeze the early convolutional layers (optional but recommended for small datasets)
    # This keeps the foundational edge/texture detection weights locked
    for param in model.features[:15].parameters():
        param.requires_grad = False

    # 3. Replace the final classification head
    # MobileNetV2 stores its classifier in model.classifier[1]
    in_features = model.last_channel
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

# Initialize Model 1 (Gatekeeper: 3 Classes)
print(f"Building Gatekeeper Model for classes: {gatekeeper_classes}")
gatekeeper_model = build_model(num_classes=len(gatekeeper_classes))
gatekeeper_model = gatekeeper_model.to(device) # Move to GPU

# Initialize Model 2 (Severity Assessor: 3 Classes)
print(f"Building Severity Model for classes: {severity_classes}")
severity_model = build_model(num_classes=len(severity_classes))
severity_model = severity_model.to(device) # Move to GPU

print("\nSuccess! Both models are built and loaded onto the GPU.")

Building Gatekeeper Model for classes: ['Eczema', 'Melanoma', 'Psoriasis']
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 85.1MB/s]


Building Severity Model for classes: ['Atopic dermatitis childhood phase', 'Atopic dermatitis feet', 'Dyshidrosis', 'Eczema areola', 'Eczema asteatotic', 'Eczema chronic', 'Eczema fingertips', 'Eczema foot', 'Eczema hand', 'Ichthyosis', 'Keratolysis exfoliativa', 'Keratosis pilaris', 'Lichen simplex chronicus', 'Neurotic excoriations', 'Pompholyx', 'Prurigo nodularis', 'Stasis dermatitis']

Success! Both models are built and loaded onto the GPU.


In [6]:
import torch.optim as optim
from torch.optim import lr_scheduler

# 1. Loss Function (Criterion)
criterion = nn.CrossEntropyLoss()

# 2. Optimizers
# We filter the parameters so the optimizer only updates the unfrozen layers
gatekeeper_optimizer = optim.Adam(filter(lambda p: p.requires_grad, gatekeeper_model.parameters()), lr=0.001)
severity_optimizer = optim.Adam(filter(lambda p: p.requires_grad, severity_model.parameters()), lr=0.001)

# 3. Learning Rate Schedulers
# Reduces the learning rate by a factor of 0.1 if the validation loss doesn't improve for 3 epochs
gatekeeper_scheduler = lr_scheduler.ReduceLROnPlateau(gatekeeper_optimizer, mode='min', factor=0.1, patience=3)
severity_scheduler = lr_scheduler.ReduceLROnPlateau(severity_optimizer, mode='min', factor=0.1, patience=3)

print("Loss functions, optimizers, and schedulers are locked in!")

Loss functions, optimizers, and schedulers are locked in!


In [9]:
import time
import copy
import torch

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=10):
    since = time.time()

    # Keep a copy of the best weights so far
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 15)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
                dataloader = train_loader
            else:
                model.eval()   # Set model to evaluate mode
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data in batches
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                # Track history only if in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Calculate statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Step the learning rate scheduler based on validation loss, and save the best model
            if phase == 'val':
                scheduler.step(epoch_loss)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Accuracy: {best_acc:.4f}')

    # Load best model weights before returning
    model.load_state_dict(best_model_wts)
    return model

In [10]:
import torch

print("Starting Training for Gatekeeper Model (Eczema vs Melanoma vs Psoriasis)...")
EPOCHS = 10  # Start with 10 for testing; you can increase this to 20-30 later for better accuracy

gatekeeper_model = train_model(
    model=gatekeeper_model,
    train_loader=gatekeeper_train_loader,
    val_loader=gatekeeper_val_loader,
    criterion=criterion,
    optimizer=gatekeeper_optimizer,
    scheduler=gatekeeper_scheduler,
    num_epochs=EPOCHS
)

# Save the model weights and the class mappings together
gatekeeper_save_path = '/content/gatekeeper_model.pth'
torch.save({
    'model_state_dict': gatekeeper_model.state_dict(),
    'classes': gatekeeper_classes
}, gatekeeper_save_path)

print(f"Gatekeeper Model saved successfully to {gatekeeper_save_path}")

Starting Training for Gatekeeper Model (Eczema vs Melanoma vs Psoriasis)...
Epoch 1/10
---------------
Train Loss: 0.3864 Acc: 0.8015
Val Loss: 0.3325 Acc: 0.8320

Epoch 2/10
---------------
Train Loss: 0.3200 Acc: 0.8395
Val Loss: 0.2857 Acc: 0.8458

Epoch 3/10
---------------
Train Loss: 0.2700 Acc: 0.8741
Val Loss: 0.2985 Acc: 0.8676

Epoch 4/10
---------------
Train Loss: 0.2278 Acc: 0.8976
Val Loss: 0.3135 Acc: 0.8633

Epoch 5/10
---------------
Train Loss: 0.2163 Acc: 0.9114
Val Loss: 0.3149 Acc: 0.8727

Epoch 6/10
---------------
Train Loss: 0.1917 Acc: 0.9167
Val Loss: 0.3249 Acc: 0.8756

Epoch 7/10
---------------
Train Loss: 0.1262 Acc: 0.9498
Val Loss: 0.2921 Acc: 0.8989

Epoch 8/10
---------------
Train Loss: 0.1112 Acc: 0.9567
Val Loss: 0.2948 Acc: 0.8967

Epoch 9/10
---------------
Train Loss: 0.1044 Acc: 0.9613
Val Loss: 0.3085 Acc: 0.8953

Epoch 10/10
---------------
Train Loss: 0.0945 Acc: 0.9654
Val Loss: 0.3411 Acc: 0.8967

Training complete in 6m 16s
Best Validation

In [11]:
import torch
from google.colab import files

print("Starting Training for Severity Model (Low vs Moderate vs High)...")

severity_model = train_model(
    model=severity_model,
    train_loader=severity_train_loader,
    val_loader=severity_val_loader,
    criterion=criterion,
    optimizer=severity_optimizer,
    scheduler=severity_scheduler,
    num_epochs=EPOCHS
)

# Save the model weights and the class mappings together
severity_save_path = '/content/severity_model.pth'
torch.save({
    'model_state_dict': severity_model.state_dict(),
    'classes': severity_classes
}, severity_save_path)

print(f"Severity Model saved successfully to {severity_save_path}")

# Trigger downloads to your local desktop
print("Downloading models to your local machine...")
files.download('/content/gatekeeper_model.pth')
files.download('/content/severity_model.pth')

Starting Training for Severity Model (Low vs Moderate vs High)...
Epoch 1/10
---------------
Train Loss: 1.8051 Acc: 0.4561
Val Loss: 1.4590 Acc: 0.5735

Epoch 2/10
---------------
Train Loss: 1.1915 Acc: 0.6039
Val Loss: 1.1858 Acc: 0.6380

Epoch 3/10
---------------
Train Loss: 0.8654 Acc: 0.7204
Val Loss: 1.2619 Acc: 0.6595

Epoch 4/10
---------------
Train Loss: 0.7945 Acc: 0.7401
Val Loss: 1.3275 Acc: 0.6272

Epoch 5/10
---------------
Train Loss: 0.5836 Acc: 0.8038
Val Loss: 1.2545 Acc: 0.6810

Epoch 6/10
---------------
Train Loss: 0.5489 Acc: 0.8127
Val Loss: 1.3592 Acc: 0.6344

Epoch 7/10
---------------
Train Loss: 0.3735 Acc: 0.8701
Val Loss: 1.0998 Acc: 0.7061

Epoch 8/10
---------------
Train Loss: 0.2557 Acc: 0.9203
Val Loss: 1.0919 Acc: 0.7204

Epoch 9/10
---------------
Train Loss: 0.2373 Acc: 0.9158
Val Loss: 1.0904 Acc: 0.7240

Epoch 10/10
---------------
Train Loss: 0.2008 Acc: 0.9337
Val Loss: 1.0442 Acc: 0.7455

Training complete in 1m 28s
Best Validation Accuracy:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>